# 4. Classificação pelos eixos da BNCC
A classificação é baseada na presença de termos e expressões no texto normalizado. Um artigo pode receber mais de um eixo. Todas as evidências serão registradas para revisão.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import carregar_eixos_bncc, selecionar_descritores_relevantes, encontrar_evidencias_bncc
pasta_processados = raiz / 'dados' / '1_processados'

## 4.1 Leitura dos artigos e do vocabulário

In [ ]:
# Carregar os dados processados
df = pd.read_csv(pasta_processados / '02_artigos_pre_processados.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
eixos_candidatos = carregar_eixos_bncc(raiz / 'apoio' / 'termos_bncc.yml')
eixos = selecionar_descritores_relevantes(
    eixos_candidatos,
    set(ranking_termos['termo']),
    set(ranking_bigramas['bigrama']),
)
print(f'Artigos recebidos: {len(df)}')

# Criar DataFrame com os resultados
pd.DataFrame([
    {'codigo': codigo, 'eixo': dados['nome'], 'termos_ativos': len(dados['termos']), 'bigramas_ativos': len(dados['bigramas'])}
    for codigo, dados in eixos.items()
])

## 4.2 Aplicação das regras
A busca usa palavras e expressões completas. Por exemplo, o termo `tic` não será encontrado dentro de outra palavra.

In [ ]:
df['evidencias_bncc'] = df['texto_limpo'].fillna('').apply(
    lambda texto: encontrar_evidencias_bncc(texto, eixos)
)
df['eixos_bncc'] = df['evidencias_bncc'].apply(
    lambda resultado: '; '.join(eixos[codigo]['nome'] for codigo in resultado)
)
df['quantidade_eixos'] = df['evidencias_bncc'].apply(len)
df[['id_artigo', 'titulo', 'eixos_bncc', 'quantidade_eixos']].head(10)

## 4.3 Tabela de classificações e evidências

In [ ]:
linhas_classificacao = []
# Gerar linhas de classificação para cada artigo e cada eixo
for _, artigo in df.iterrows():
    for codigo_eixo, evidencias in artigo['evidencias_bncc'].items():
        linhas_classificacao.append({
            'id_artigo': artigo['id_artigo'],
            'evento': artigo['evento'],
            'ano': artigo['ano'],
            'eixo_bncc': eixos[codigo_eixo]['nome'],
            'quantidade_evidencias': len(evidencias['termos']) + len(evidencias['bigramas']),
            'termos_encontrados': ' | '.join(evidencias['termos']),
            'bigramas_encontrados': ' | '.join(evidencias['bigramas']),
        })

# Criar DataFrame com as classificações
classificacoes = pd.DataFrame(linhas_classificacao)
classificacoes.head(10)

## 4.4 Resumo e cobertura da classificação

In [ ]:
# Gerar resumo por eixo
resumo_eixos = (classificacoes.groupby('eixo_bncc')['id_artigo']
    .nunique()
    .rename('quantidade_artigos')
    .reset_index())
resumo_eixos['percentual_corpus'] = (resumo_eixos['quantidade_artigos'] / len(df) * 100).round(2)
resumo_eixos.sort_values('quantidade_artigos', ascending=False)

In [ ]:
# Gerar resumo por quantidade de eixos
resumo_sobreposicao = (df['quantidade_eixos']
    .value_counts()
    .sort_index()
    .rename_axis('quantidade_eixos')
    .reset_index(name='quantidade_artigos'))
resumo_sobreposicao['percentual_corpus'] = (resumo_sobreposicao['quantidade_artigos'] / len(df) * 100).round(2)
resumo_sobreposicao

## 4.5 Artigos não classificados

In [ ]:
nao_classificados = df[df['quantidade_eixos'] == 0].copy()
print(f'Não classificados: {len(nao_classificados)} de {len(df)}')
nao_classificados[['id_artigo', 'evento', 'ano', 'titulo']].head(20)

## 4.6 Descritores que mais produziram classificações
Esta tabela ajuda a identificar termos excessivamente amplos ou pouco utilizados.

In [ ]:
evidencias_termos = (classificacoes[['id_artigo', 'eixo_bncc', 'termos_encontrados']]
    .assign(tipo='termo', descritor=lambda x: x['termos_encontrados'].fillna('').str.split(' | ', regex=False))
    .explode('descritor').query("descritor != ''"))
evidencias_bigramas = (classificacoes[['id_artigo', 'eixo_bncc', 'bigramas_encontrados']]
    .assign(tipo='bigrama', descritor=lambda x: x['bigramas_encontrados'].fillna('').str.split(' | ', regex=False))
    .explode('descritor').query("descritor != ''"))
evidencias_descritores = pd.concat([
    evidencias_termos[['id_artigo', 'eixo_bncc', 'tipo', 'descritor']],
    evidencias_bigramas[['id_artigo', 'eixo_bncc', 'tipo', 'descritor']],
])
frequencia_descritores = (evidencias_descritores.groupby(['eixo_bncc', 'tipo', 'descritor'])['id_artigo']
    .nunique()
    .rename('quantidade_artigos')
    .reset_index()
    .sort_values(['eixo_bncc', 'quantidade_artigos'], ascending=[True, False]))
frequencia_descritores.groupby('eixo_bncc').head(10)

## 4.7 Exportação para validação

In [ ]:
artigos_classificados = df.drop(columns='evidencias_bncc')
artigos_classificados.to_csv(pasta_processados / '04_artigos_classificados_bncc.csv', index=False, encoding='utf-8-sig')
classificacoes.to_csv(pasta_processados / '04_classificacoes_bncc.csv', index=False, encoding='utf-8-sig')
nao_classificados.drop(columns='evidencias_bncc').to_csv(pasta_processados / '04_artigos_nao_classificados.csv', index=False, encoding='utf-8-sig')
resumo_eixos.to_csv(pasta_processados / '04_resumo_classificacao_bncc.csv', index=False, encoding='utf-8-sig')
frequencia_descritores.to_csv(pasta_processados / '04_frequencia_descritores_bncc.csv', index=False, encoding='utf-8-sig')
print('Arquivos de validação exportados.')